# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminahamamdzic/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (anonymized), aggregated over the observation window. The window runs from [FILL START] to [FILL END] ([FILL N] days). Grain and window are both verified with queries in section 3 below. It is not one query and not one client per row — one page over that window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
import os, subprocess
import pandas as pd

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
]
path = next((p for p in candidates if os.path.exists(p)), None)

if path is None:
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"],
        check=True,
    )
    path = "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)
print("Loaded from:", path)
print("Shape (rows, cols):", df.shape)

Loaded from: flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Shape (rows, cols): (30000, 44)


In [4]:
print("Columns + dtypes:")
print(df.dtypes)

Columns + dtypes:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            in

- Feature (goes into the model): impressions, clicks, CTR, average position, position trend / change, page age or days-since-update, word count, internal links, sessions, scroll/engagement — the measurable signals of how the page performs.
- Label (what I predict): the needs_refresh proxy flag defined in 01_prepare_features.py. One value per page. This is the target, so it never goes in as a feature.
- Context (kept, but not trained on): anonymized page id, anonymized client id, and the window start/end dates. I keep these to group, join, and split by client — not as inputs.
- Excluded (with why): raw URL / title / query text → privacy (never in output). Any post-window outcome field → leakage (it encodes the label). Google-anonymized / rare-tail query counts → unreliable (systematically undercounted). GA4/engagement fields on early rows → too sparse to trust (see section 4).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
import pandas as pd

# --- grain: is one row really one page? ---
print("Rows, cols:", df.shape)
id_candidates = [c for c in df.columns
                 if any(k in c.lower() for k in ["page", "url", "content", "id"])]
print("Likely page-id column(s):", id_candidates)
if id_candidates:
    idc = id_candidates[0]
    print(f"Unique {idc}: {df[idc].nunique()} of {len(df)} rows")
    print(f"Duplicate rows on {idc}: {df.duplicated(subset=[idc]).sum()}")

# --- counts + missing values ---
miss = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("\nMissing % (top 15):")
print(miss.head(15))

# --- time window ---
date_cols = [c for c in df.columns
             if any(k in c.lower() for k in ["date", "day", "month", "start", "end", "period"])]
print("\nLikely date column(s):", date_cols)
for c in date_cols:
    d = pd.to_datetime(df[c], errors="coerce")
    if d.notna().any():
        print(f"  {c}: {d.min().date()} -> {d.max().date()}")

Rows, cols: (30000, 44)
Likely page-id column(s): ['content_id', 'client_id', 'content_type', 'provider_used', 'pageviews_90d', 'content_age_days']
Unique content_id: 30000 of 30000 rows
Duplicate rows on content_id: 0

Missing % (top 15):
provider_used        71.5
word_count           25.7
char_count           25.7
word_count_tier      25.7
char_count_tier      25.7
model_used           19.1
trend_pct            11.3
competition_level     8.7
search_volume         8.2
cpc                   8.2
competition           8.2
main_intent           7.9
scroll_rate           0.4
content_type          0.0
client_id             0.0
dtype: float64

Likely date column(s): ['days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'trend_direction', 'trend_pct']
  days_with_impressions: 1970-01-01 -> 1970-01-01
  days_with_sessions: 1970-01-01 -> 1970-01-01
  content_age_days: 1970-01-01 -> 1970-01-01
  days_since_last_update: 1970-01-01 -> 1970-01-01
  trend_pct:

/tmp/ipykernel_1212/3556689467.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d = pd.to_datetime(df[c], errors="coerce")


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Observational, not causal. The data shows what pages did, never why, and can't tell me whether a refresh would actually recover traffic. Every result is directional / decision-support.
- Unbalanced history. Client history depth differs, so early periods are thinner and per-client comparisons aren't apples-to-apples.
- GSC-only early rows. Early rows carry search metrics (impressions, clicks, position) but not analytics/GA4 metrics (sessions, scroll), so engagement features there are missing, not zero.
- Overlapping windows. Last-30 / prev-30 sub-windows sit inside the same 90-day span, so trend features aren't independent samples.
- Anonymized queries. Google omits low-volume/rare queries, so query-level counts undercount and won't reconcile to page totals.
- Proxy label, no ground truth. "Needs refresh" is a defined rule, not an observed outcome — my metric measures agreement with that rule, not real-world impact.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.